In [1]:
import pandas as pd
import json

df = pd.read_excel('../../../data/Elsevier-LIS/Texts-lite-abstract.xlsx')
links = df['Pii'].tolist()
abs = df['reserve_4'].tolist()
hts = df['Highlights'].tolist()

link_to_keywords = {}
with open('../../../data/Elsevier-LIS/Keywords.json', 'r') as f:
    link_to_keywords = json.load(f)

keywords = []
for i, link in enumerate(links):
    try:
        keywords.append(link_to_keywords[link])
    except:
        keywords.append([])

print(len(keywords))

2589


In [2]:
import nltk
import numpy as np

porter = nltk.PorterStemmer()
def stemmer(raw_sequences):
    stemmed_sequences = []

    for i, words in enumerate(raw_sequences):
        new_words = []
        for word in words:
            if type(word) == list:
                # for h_candidates and a_candidates
                word = word[0]
            items = word.split()
            new_word = ' '.join(porter.stem(item) for item in items)
            new_words.append(new_word.strip())
        stemmed_sequences.append(new_words)
    
    return stemmed_sequences


def calPRF(num_c, num_e, num_s):
    F1 = 0.0
    P = float(num_c) / float(num_e) if num_e!=0 else 0.0
    R = float(num_c) / float(num_s) if num_s!=0 else 0.0
    if (P + R == 0.0):
        F1 = 0
    else:
        F1 = 2 * P * R / (P + R)
    return P, R, F1


def getPRF_docwise(references, predictions, log=None):
    """
    按“逐文档”计算 P/R/F1：
    - references: list[list[str]]，每个文档的真实关键词
    - predictions: list[list[str]]，每个文档的预测关键词（按排序）
    
    返回：
    - per_doc_metrics: {
        "P5": [..], "R5": [..], "F15": [..] 等
      }
    同时可在 log 中输出平均值。
    """

    P5_list, R5_list, F5_list = [], [], []
    P10_list, R10_list, F10_list = [], [], []
    P15_list, R15_list, F15_list = [], [], []

    assert len(references) == len(predictions), "refs 和 preds 数量必须一致"

    for i in range(len(references)):
        reference = references[i]
        prediction = predictions[i]

        num_s = len(reference)  # 当前文档真实关键词数量

        # --- k = 5 ---
        pred_k = prediction[:5]
        num_e_5 = len(pred_k)
        num_c_5 = sum(1 for cand in pred_k if cand in reference)
        P5, R5, F5 = calPRF(num_c_5, num_e_5, num_s)
        P5_list.append(P5)
        R5_list.append(R5)
        F5_list.append(F5)

        # --- k = 10 ---
        pred_k = prediction[:10]
        num_e_10 = len(pred_k)
        num_c_10 = sum(1 for cand in pred_k if cand in reference)
        P10, R10, F10 = calPRF(num_c_10, num_e_10, num_s)
        P10_list.append(P10)
        R10_list.append(R10)
        F10_list.append(F10)

        # --- k = 15 ---
        pred_k = prediction[:15]
        num_e_15 = len(pred_k)
        num_c_15 = sum(1 for cand in pred_k if cand in reference)
        P15, R15, F15 = calPRF(num_c_15, num_e_15, num_s)
        P15_list.append(P15)
        R15_list.append(R15)
        F15_list.append(F15)

    per_doc_metrics = {
        "F5": F5_list,
        "F10": F10_list,
        "F15": F15_list,
    }
    
    return per_doc_metrics

In [17]:
claude_cs = {}

In [30]:
import re
import pandas as pd

new_df = pd.read_excel('../../ModelPred/LLM_Pred_Tian/Claude/Elsevier-LIS-Claude-HA.xlsx')
raw_pred_keywords = new_df['Keywords'].tolist()
pred_keywords = []
regex = r'\d. '
for elem in raw_pred_keywords:
    keyword = []
    if ':' in elem:
        pos = elem.rindex(':')
        elem = elem[pos+1:]
        if ',' in elem:
            keyword = elem.split(',')
        elif '\n' in elem:
            keyword = elem.split('\n')
    if ':' not in elem:
        if ',' in elem:
            keyword = elem.split(',')
        elif '\n' in elem:
            keyword = elem.split('\n')

    new_keyword = []
    for i, word in enumerate(keyword):
        word = word.strip()
        if len(word.split(' ')) >=5:
            continue
        if '- ' in word:
            pos = word.index(' ')
            word = word[pos+1:]
        elif re.search(regex, word):
            pos = re.search(regex, word).span()[1]
            word = word[pos:]

        if word != '':
            if word[-1] == '.':
                new_keyword.append(word[:-1])
            else:
                new_keyword.append(word)

    keyword = [word.strip() for word in new_keyword]
    pred_keywords.append(keyword)

In [31]:
gold_standards_stem = stemmer(keywords)
pred_candidates_stem = stemmer(pred_keywords)
results = getPRF_docwise(gold_standards_stem, pred_candidates_stem)
claude_cs['HA'] = results

In [32]:
# paired t-test
from scipy import stats

a_results = claude_cs['A']

options = claude_cs.keys()
for option in options:
    if option == 'A':
        continue

    print(option)
    option_results = claude_cs[option]
    try:
        for score in ['F5', 'F10', 'F15']:
            a_result = a_results[score]
            b_result = option_results[score]
            t_stat, p_value = stats.ttest_rel(a_result, b_result)
            diff = np.mean(b_result) - np.mean(a_result)
            print(score, end=" ")
            print(diff, p_value)
    except:
        continue

FA
F5 -0.04350434767584249 2.307404535406937e-26
F10 -0.05288830520580229 1.1898883295988801e-57
F15 -0.05104420989697711 1.5070640707140525e-54
FA+H
F5 -0.022652726013096813 2.7913083510878002e-08
F10 -0.042392911367418906 3.593154280463166e-38
F15 -0.04216920883231795 1.6037123107640925e-38
H+FA
F5 -0.022652726013096813 2.7913083510878002e-08
F10 -0.042392911367418906 3.593154280463166e-38
F15 -0.04216920883231795 1.6037123107640925e-38
H
F5 -0.06740090303242102 1.6950349907459319e-47
F10 -0.048612427870828756 1.760290573866075e-34
F15 -0.04613575273900511 1.3390817203013566e-31
AH
F5 0.006668684304837247 0.06439346758277423
F10 0.007877434354838786 0.008282592820320902
F15 0.007451109535691003 0.012067921818458081
HA
F5 -0.011238105270550247 0.005891216016463822
F10 -0.0029010269160906765 0.3876034267827272
F15 -0.0024442579557575894 0.46375336576044934
